# NeuroGS-Codec: Topology-aware Rate–Distortion 3D Gaussian Volume Codec (Starter Notebook)

**Purpose:** End-to-end, ready-to-run baseline for compressing a **3D microscopy volume** (voxel intensities) using an anisotropic 3D Gaussian mixture with:
- neurite-biased **structure weighting** (tubularity proxy)
- **rate–distortion** training with quantization-in-the-loop (STE)
- optional **densify/prune** hooks (microscopy-aware)

**Created:** 2026-02-03 09:14:53

> This notebook is a clean research scaffold. It prioritizes clarity over maximum speed.


## 0) Setup

In [3]:
!pip install tifffile

In [1]:

# If running in Colab, uncomment:
# !pip -q install tifffile scipy tqdm

import os
import math
import time
from dataclasses import dataclass
from typing import Tuple, Dict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    import tifffile as tiff
except ImportError as e:
    raise ImportError("Please install tifffile: pip install tifffile") from e

from tqdm.auto import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_float32_matmul_precision("high")
print("Device:", DEVICE)


/home/armin/miniconda3/envs/neurogs/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


## 1) Load a 3D volume (.tif stack)

In [ ]:

# ---- User config ----
TIF_PATH = "YOUR_VOLUME.tif"   # <-- set this
# Example anisotropic spacing (microns). Replace with your microscope metadata.
VOXEL_SPACING = (0.126, 0.126, 1.0)  # (dx, dy, dz)
# ---------------------

assert os.path.exists(TIF_PATH), f"File not found: {TIF_PATH}"

V_np = tiff.imread(TIF_PATH)  # expected shape: (Z, Y, X)
assert V_np.ndim == 3, f"Expected 3D volume, got shape {V_np.shape}"

print("Loaded:", V_np.shape, V_np.dtype, "min/max:", float(V_np.min()), float(V_np.max()))

# Normalize to [0,1] float32
V = V_np.astype(np.float32)
V = (V - V.min()) / (V.max() - V.min() + 1e-8)
V_t = torch.from_numpy(V).to(DEVICE)  # (Z,Y,X)


## 2) Build physical coordinate grid (anisotropy-aware)

In [ ]:

def make_coord_grid_zyx(shape_zyx: Tuple[int,int,int], spacing_xyz: Tuple[float,float,float], device: str):
    # Returns normalized physical coordinates in [-1,1]^3 for each voxel index.
    # shape_zyx: (Z,Y,X), spacing_xyz: (dx,dy,dz)
    Z, Y, X = shape_zyx
    dx, dy, dz = spacing_xyz

    xs = torch.arange(X, device=device) * dx
    ys = torch.arange(Y, device=device) * dy
    zs = torch.arange(Z, device=device) * dz

    x0, x1 = xs.min(), xs.max()
    y0, y1 = ys.min(), ys.max()
    z0, z1 = zs.min(), zs.max()

    xn = (xs - x0) / (x1 - x0 + 1e-8) * 2 - 1
    yn = (ys - y0) / (y1 - y0 + 1e-8) * 2 - 1
    zn = (zs - z0) / (z1 - z0 + 1e-8) * 2 - 1

    zz, yy, xx = torch.meshgrid(zn, yn, xn, indexing="ij")
    coords = torch.stack([xx, yy, zz], dim=-1)  # (Z,Y,X,3) in (x,y,z)
    return coords

coords_grid = make_coord_grid_zyx(V_t.shape, VOXEL_SPACING, DEVICE)
print("coords_grid:", coords_grid.shape, coords_grid.dtype)


## 3) Neurite-likelihood map M(x): simple tubularity proxy
Label-free prior derived from the volume (DoG + gradient magnitude).

In [ ]:

def gaussian_blur_3d(vol: torch.Tensor, sigma: float) -> torch.Tensor:
    # Separable 3D Gaussian blur using 1D kernels and conv3d.
    if sigma <= 0:
        return vol
    radius = int(3 * sigma + 0.5)
    x = torch.arange(-radius, radius+1, device=vol.device, dtype=vol.dtype)
    k = torch.exp(-(x**2)/(2*sigma**2))
    k = k / (k.sum() + 1e-8)

    v = vol.unsqueeze(0).unsqueeze(0)  # (1,1,Z,Y,X)

    kx = k.view(1,1,1,1,-1)
    v = F.conv3d(v, kx, padding=(0,0,radius))

    ky = k.view(1,1,1,-1,1)
    v = F.conv3d(v, ky, padding=(0,radius,0))

    kz = k.view(1,1,-1,1,1)
    v = F.conv3d(v, kz, padding=(radius,0,0))

    return v[0,0]

@torch.no_grad()
def make_neurite_map(vol_zyx: torch.Tensor) -> torch.Tensor:
    v1 = gaussian_blur_3d(vol_zyx, sigma=0.8)
    v2 = gaussian_blur_3d(vol_zyx, sigma=2.0)
    dog = (v1 - v2).abs()

    dz = F.pad(vol_zyx[1:] - vol_zyx[:-1], (0,0,0,0,0,1))
    dy = F.pad(vol_zyx[:,1:] - vol_zyx[:,:-1], (0,0,0,1,0,0))
    dx = F.pad(vol_zyx[:,:,1:] - vol_zyx[:,:,:-1], (0,1,0,0,0,0))
    gmag = torch.sqrt(dx*dx + dy*dy + dz*dz + 1e-8)

    m = dog + 0.5 * gmag
    m = (m - m.min()) / (m.max() - m.min() + 1e-8)
    return m.clamp(0,1)

M = make_neurite_map(V_t)
print("M:", M.shape, "min/max:", float(M.min()), float(M.max()), "mean:", float(M.mean()))


## 4) Sampling strategy (uniform + neurite-biased)

In [ ]:

@torch.no_grad()
def sample_points(coords_grid: torch.Tensor, V: torch.Tensor, M: torch.Tensor,
                  n_uniform: int, n_biased: int):
    Z,Y,X,_ = coords_grid.shape

    idx_u = torch.randint(0, Z*Y*X, (n_uniform,), device=V.device)
    zu = idx_u // (Y*X)
    yu = (idx_u % (Y*X)) // X
    xu = idx_u % X

    flatM = M.reshape(-1)
    probs = flatM / (flatM.sum() + 1e-8)
    idx_b = torch.multinomial(probs, n_biased, replacement=True)
    zb = idx_b // (Y*X)
    yb = (idx_b % (Y*X)) // X
    xb = idx_b % X

    z = torch.cat([zu, zb], dim=0)
    y = torch.cat([yu, yb], dim=0)
    x = torch.cat([xu, xb], dim=0)

    pts = coords_grid[z,y,x]     # (N,3)
    tgt = V[z,y,x]               # (N,)
    mval = M[z,y,x]              # (N,)
    return pts, tgt, mval

pts, tgt, mval = sample_points(coords_grid, V_t, M, 1024, 1024)
print("Sampled:", pts.shape, tgt.shape, "m mean:", float(mval.mean()))


## 5) Gaussian mixture model (anisotropic, rotation via quaternion)

In [ ]:

def quat_to_rotmat(q: torch.Tensor) -> torch.Tensor:
    # q: (...,4) in (w,x,y,z), assumed normalized
    w,x,y,z = q.unbind(-1)
    ww,xx,yy,zz = w*w, x*x, y*y, z*z
    wx,wy,wz = w*x, w*y, w*z
    xy,xz,yz = x*y, x*z, y*z

    R = torch.stack([
        ww+xx-yy-zz, 2*(xy-wz),     2*(xz+wy),
        2*(xy+wz),   ww-xx+yy-zz,   2*(yz-wx),
        2*(xz-wy),   2*(yz+wx),     ww-xx-yy+zz
    ], dim=-1).reshape(q.shape[:-1]+(3,3))
    return R

def safe_normalize(q: torch.Tensor, eps=1e-8) -> torch.Tensor:
    return q / (q.norm(dim=-1, keepdim=True) + eps)

class GaussianMixtureVolume(nn.Module):
    def __init__(self, N: int, init_means: torch.Tensor, init_amp: torch.Tensor):
        super().__init__()
        assert init_means.shape == (N,3)
        assert init_amp.shape == (N,)
        self.N = N

        self.mu = nn.Parameter(init_means.clone())
        self.log_s = nn.Parameter(torch.zeros(N,3, device=init_means.device) - 2.0)

        q = torch.zeros(N,4, device=init_means.device)
        q[:,0] = 1.0
        self.q = nn.Parameter(q)

        self.a = nn.Parameter(init_amp.clone())
        self.b = nn.Parameter(torch.tensor(0.0, device=init_means.device))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        mu = self.mu[None, :, :]                 # (1,N,3)
        dx = x[:,None,:] - mu                    # (P,N,3)

        s = torch.exp(self.log_s).clamp(1e-4, 10.0)  # (N,3)
        qn = safe_normalize(self.q)
        R = quat_to_rotmat(qn)                   # (N,3,3)
        Rt = R.transpose(-1,-2)

        y = torch.einsum("pni,nij->pnj", dx, Rt) # (P,N,3)
        y = y / (s[None,:,:] + 1e-8)
        exp_term = -0.5 * (y*y).sum(dim=-1)      # (P,N)
        g = torch.exp(exp_term)                  # (P,N)

        pred = (g * self.a[None,:]).sum(dim=1) + self.b
        return pred

@torch.no_grad()
def init_gaussians_from_neurite_map(coords_grid, V, M, N0: int):
    Z,Y,X,_ = coords_grid.shape
    flatM = M.reshape(-1)
    probs = flatM / (flatM.sum() + 1e-8)
    idx = torch.multinomial(probs, N0, replacement=True)
    z = idx // (Y*X)
    y = (idx % (Y*X)) // X
    x = idx % X
    means = coords_grid[z,y,x]
    amp = V[z,y,x].clone()
    return means, amp

N0 = 6000  # start modest; add densify later
init_means, init_amp = init_gaussians_from_neurite_map(coords_grid, V_t, M, N0)
model = GaussianMixtureVolume(N0, init_means, init_amp).to(DEVICE)
print("Model N:", model.N)


## 6) Rate model (differentiable bitrate proxy)
Quantization-in-the-loop uses STE.

In [ ]:

class LaplaceEntropyModel(nn.Module):
    def __init__(self, init_scale=1.0):
        super().__init__()
        self.log_b = nn.Parameter(torch.tensor(math.log(init_scale), device=DEVICE))

    def nll_bits(self, xq: torch.Tensor) -> torch.Tensor:
        b = torch.exp(self.log_b).clamp(1e-6, 1e3)
        logp = -math.log(2.0) - torch.log(b) - (xq.abs() / b)
        bits = (-logp / math.log(2.0)).mean()
        return bits

def ste_round(x: torch.Tensor) -> torch.Tensor:
    return (x.round() - x).detach() + x

@dataclass
class QuantSteps:
    mu: float = 1/2048
    log_s: float = 1/256
    q: float = 1/1024
    a: float = 1/1024
    b: float = 1/1024

H_mu   = LaplaceEntropyModel(init_scale=0.2).to(DEVICE)
H_logs = LaplaceEntropyModel(init_scale=0.5).to(DEVICE)
H_q    = LaplaceEntropyModel(init_scale=0.2).to(DEVICE)
H_a    = LaplaceEntropyModel(init_scale=0.5).to(DEVICE)
H_b    = LaplaceEntropyModel(init_scale=0.5).to(DEVICE)

Q = QuantSteps()


## 7) Losses: distortion + topology proxy + rate

In [ ]:

def charbonnier(x, eps=1e-3):
    return torch.sqrt(x*x + eps*eps)

@torch.no_grad()
def extract_random_patch(vol: torch.Tensor, patch_zyx=(32,64,64)):
    Z,Y,X = vol.shape
    pz,py,px = patch_zyx
    z0 = torch.randint(0, max(1, Z-pz+1), (1,), device=vol.device).item()
    y0 = torch.randint(0, max(1, Y-py+1), (1,), device=vol.device).item()
    x0 = torch.randint(0, max(1, X-px+1), (1,), device=vol.device).item()
    return (z0,y0,x0), vol[z0:z0+pz, y0:y0+py, x0:x0+px]

def tv3d(p: torch.Tensor):
    dz = (p[1:] - p[:-1]).abs().mean()
    dy = (p[:,1:] - p[:,:-1]).abs().mean()
    dx = (p[:,:,1:] - p[:,:,:-1]).abs().mean()
    return dx + dy + dz

def patch_topology_loss(model: GaussianMixtureVolume, coords_grid: torch.Tensor, V: torch.Tensor,
                        patch_zyx=(16,32,32), tau=0.25, gamma=10.0):
    (z0,y0,x0), _ = extract_random_patch(V, patch_zyx)
    pz,py,px = patch_zyx
    coords = coords_grid[z0:z0+pz, y0:y0+py, x0:x0+px].reshape(-1,3)
    pred = model(coords).reshape(pz,py,px)

    P = torch.sigmoid(gamma*(pred - tau))
    tv = tv3d(P)

    P_ = P[None,None]
    low = F.avg_pool3d(P_, kernel_size=3, stride=1, padding=1)[0,0]
    hf = (P - low).abs().mean()
    return tv + 0.5*hf


## 8) Training loop (RD + topology)

In [ ]:

def compute_rate_bits(model: GaussianMixtureVolume) -> torch.Tensor:
    mu_q   = ste_round(model.mu / Q.mu)
    logs_q = ste_round(model.log_s / Q.log_s)
    q_q    = ste_round(model.q / Q.q)
    a_q    = ste_round(model.a / Q.a)
    b_q    = ste_round(model.b / Q.b)

    bits = (H_mu.nll_bits(mu_q) +
            H_logs.nll_bits(logs_q) +
            H_q.nll_bits(q_q) +
            H_a.nll_bits(a_q) +
            H_b.nll_bits(b_q))
    return bits

def train(model: GaussianMixtureVolume,
          V: torch.Tensor, coords_grid: torch.Tensor, M: torch.Tensor,
          steps=2000, batch=120_000,
          kappa=4.0, lam=0.06, alpha=0.04,
          topo_every=20, topo_patch=(16,32,32),
          lr=2e-2):
    params = list(model.parameters()) + list(H_mu.parameters()) + list(H_logs.parameters()) + list(H_q.parameters()) + list(H_a.parameters()) + list(H_b.parameters())
    opt = torch.optim.Adam(params, lr=lr)

    losses = {"D": [], "R": [], "T": [], "Total": []}
    t0 = time.time()

    for it in tqdm(range(steps), desc="Training"):
        n_u = batch//2
        n_b = batch - n_u
        pts, tgt, mval = sample_points(coords_grid, V, M, n_u, n_b)

        pred = model(pts)
        w = 1.0 + kappa * mval
        D = (w * charbonnier(pred - tgt)).mean()

        R = compute_rate_bits(model)

        T = torch.tensor(0.0, device=V.device)
        if (it % topo_every) == 0:
            T = patch_topology_loss(model, coords_grid, V, patch_zyx=topo_patch)

        total = D + lam * R + alpha * T

        opt.zero_grad(set_to_none=True)
        total.backward()
        opt.step()

        losses["D"].append(float(D.detach().cpu()))
        losses["R"].append(float(R.detach().cpu()))
        losses["T"].append(float(T.detach().cpu()))
        losses["Total"].append(float(total.detach().cpu()))

        if (it+1) % 200 == 0:
            dt = time.time() - t0
            print(f"iter {it+1:5d} | D={losses['D'][-1]:.5f} R={losses['R'][-1]:.3f} T={losses['T'][-1]:.4f} Tot={losses['Total'][-1]:.5f} | {dt:.1f}s")

    return losses

losses = train(model, V_t, coords_grid, M)


## 9) Full-volume reconstruction

In [ ]:

@torch.no_grad()
def reconstruct_full(model: GaussianMixtureVolume, coords_grid: torch.Tensor, chunk=800_000):
    Z,Y,X,_ = coords_grid.shape
    coords = coords_grid.reshape(-1,3)
    out = torch.empty(coords.shape[0], device=coords.device, dtype=torch.float32)
    for s in tqdm(range(0, coords.shape[0], chunk), desc="Reconstruct"):
        out[s:s+chunk] = model(coords[s:s+chunk])
    return out.reshape(Z,Y,X).clamp(0,1)

V_hat = reconstruct_full(model, coords_grid)
print("Reconstructed:", V_hat.shape, "min/max:", float(V_hat.min()), float(V_hat.max()))


## 10) Metrics: PSNR/SSIM + neurite-map consistency (proxy)

In [ ]:

@torch.no_grad()
def psnr(a: torch.Tensor, b: torch.Tensor, eps=1e-8):
    mse = ((a-b)**2).mean().clamp_min(eps)
    return float(10.0 * torch.log10(1.0 / mse).cpu())

@torch.no_grad()
def patch_ssim3d(a: torch.Tensor, b: torch.Tensor, win=7, K1=0.01, K2=0.03):
    a_ = a[None,None]
    b_ = b[None,None]
    pad = win//2
    mu_a = F.avg_pool3d(a_, win, stride=1, padding=pad)
    mu_b = F.avg_pool3d(b_, win, stride=1, padding=pad)
    sigma_a = F.avg_pool3d(a_*a_, win, stride=1, padding=pad) - mu_a*mu_a
    sigma_b = F.avg_pool3d(b_*b_, win, stride=1, padding=pad) - mu_b*mu_b
    sigma_ab= F.avg_pool3d(a_*b_, win, stride=1, padding=pad) - mu_a*mu_b

    C1 = (K1**2)
    C2 = (K2**2)
    ssim_map = ((2*mu_a*mu_b + C1) * (2*sigma_ab + C2)) / ((mu_a*mu_a + mu_b*mu_b + C1) * (sigma_a + sigma_b + C2) + 1e-8)
    return float(ssim_map.mean().cpu())

PSNR = psnr(V_t, V_hat)
SSIM = patch_ssim3d(V_t, V_hat, win=7)
print("PSNR:", PSNR, "SSIM:", SSIM)

M_hat = make_neurite_map(V_hat)
m_consistency = float((M_hat - M).abs().mean().cpu())
print("Neurite-map L1 (proxy):", m_consistency)


## 11) Bitstream packing (baseline)
Quantize parameters and store as npz+gzip (starter).

In [ ]:

import gzip, io

@torch.no_grad()
def quantize_params(model: GaussianMixtureVolume) -> Dict[str, np.ndarray]:
    mu   = (model.mu / Q.mu).round().to(torch.int32).cpu().numpy()
    logs = (model.log_s / Q.log_s).round().to(torch.int16).cpu().numpy()
    q    = (safe_normalize(model.q) / Q.q).round().to(torch.int16).cpu().numpy()
    a    = (model.a / Q.a).round().to(torch.int16).cpu().numpy()
    b    = (model.b / Q.b).round().to(torch.int16).cpu().numpy()
    return {"mu": mu, "log_s": logs, "q": q, "a": a, "b": b,
            "Q_mu": np.array([Q.mu], np.float32),
            "Q_log_s": np.array([Q.log_s], np.float32),
            "Q_q": np.array([Q.q], np.float32),
            "Q_a": np.array([Q.a], np.float32),
            "Q_b": np.array([Q.b], np.float32)
           }

def save_bitstream_npz_gz(params: Dict[str,np.ndarray], out_path: str):
    bio = io.BytesIO()
    np.savez_compressed(bio, **params)
    raw = bio.getvalue()
    with gzip.open(out_path, "wb", compresslevel=9) as f:
        f.write(raw)
    return len(raw), os.path.getsize(out_path)

OUT_BITSTREAM = "neurogs_codec_stream.npz.gz"
raw_sz, gz_sz = save_bitstream_npz_gz(quantize_params(model), OUT_BITSTREAM)
print("Saved:", OUT_BITSTREAM)
print("Raw npz bytes:", raw_sz, "Gzip bytes:", gz_sz)

bpp = (gz_sz * 8) / (V_t.numel())
print("Bits per voxel (bpp):", bpp)


## 12) Decode + reconstruct from bitstream (verification)

In [ ]:

@torch.no_grad()
def load_bitstream_npz_gz(path: str) -> Dict[str, np.ndarray]:
    with gzip.open(path, "rb") as f:
        raw = f.read()
    bio = io.BytesIO(raw)
    data = np.load(bio)
    return {k: data[k] for k in data.files}

@torch.no_grad()
def build_model_from_quantized(qparams: Dict[str,np.ndarray], device: str) -> GaussianMixtureVolume:
    Qmu = float(qparams["Q_mu"][0])
    Qls = float(qparams["Q_log_s"][0])
    Qq  = float(qparams["Q_q"][0])
    Qa  = float(qparams["Q_a"][0])
    Qb  = float(qparams["Q_b"][0])

    mu = torch.from_numpy(qparams["mu"]).to(device).float() * Qmu
    log_s = torch.from_numpy(qparams["log_s"]).to(device).float() * Qls
    q = torch.from_numpy(qparams["q"]).to(device).float() * Qq
    a = torch.from_numpy(qparams["a"]).to(device).float() * Qa
    b = torch.from_numpy(qparams["b"]).to(device).float().view(()) * Qb

    m = GaussianMixtureVolume(mu.shape[0], mu, a).to(device)
    m.log_s.data.copy_(log_s)
    m.q.data.copy_(q)
    m.b.data.copy_(b)
    return m

qparams = load_bitstream_npz_gz(OUT_BITSTREAM)
model_dec = build_model_from_quantized(qparams, DEVICE)

V_hat2 = reconstruct_full(model_dec, coords_grid)
print("Decode PSNR:", psnr(V_t, V_hat2), "SSIM:", patch_ssim3d(V_t, V_hat2, win=7))
print("Max abs diff between V_hat and V_hat2:", float((V_hat - V_hat2).abs().max().cpu()))


## 13) Densify / prune hooks (optional)

In [ ]:

@torch.no_grad()
def densify_stub(model: GaussianMixtureVolume, V: torch.Tensor, coords_grid: torch.Tensor, M: torch.Tensor,
                 add_count=500, jitter=0.01):
    # Minimal densify: add new Gaussians sampled from high M.
    new_mu, new_a = init_gaussians_from_neurite_map(coords_grid, V, M, add_count)
    new_mu = (new_mu + jitter*torch.randn_like(new_mu)).clamp(-1,1)

    oldN = model.N
    model.N = oldN + add_count

    model.mu = nn.Parameter(torch.cat([model.mu.data, new_mu], dim=0))
    model.log_s = nn.Parameter(torch.cat([model.log_s.data, (torch.zeros(add_count,3, device=DEVICE)-2.2)], dim=0))
    q = torch.zeros(add_count,4, device=DEVICE); q[:,0]=1
    model.q = nn.Parameter(torch.cat([model.q.data, q], dim=0))
    model.a = nn.Parameter(torch.cat([model.a.data, new_a], dim=0))
    return model

@torch.no_grad()
def prune_stub(model: GaussianMixtureVolume, amp_thresh=0.002):
    keep = (model.a.data.abs() > amp_thresh)
    if keep.sum() == keep.numel():
        return model
    model.N = int(keep.sum().item())
    model.mu = nn.Parameter(model.mu.data[keep])
    model.log_s = nn.Parameter(model.log_s.data[keep])
    model.q = nn.Parameter(model.q.data[keep])
    model.a = nn.Parameter(model.a.data[keep])
    return model

# Example usage:
# model = densify_stub(model, V_t, coords_grid, M, add_count=800)
# model = prune_stub(model, amp_thresh=0.0015)
# losses = train(model, V_t, coords_grid, M, steps=1000)


## 14) Save checkpoint

In [ ]:

CKPT_PATH = "neurogs_codec_ckpt.pt"
torch.save({
    "model_state": model.state_dict(),
    "entropy_state": {
        "H_mu": H_mu.state_dict(),
        "H_logs": H_logs.state_dict(),
        "H_q": H_q.state_dict(),
        "H_a": H_a.state_dict(),
        "H_b": H_b.state_dict(),
    },
    "Q": Q.__dict__,
    "voxel_spacing": VOXEL_SPACING,
    "shape_zyx": tuple(V_t.shape),
}, CKPT_PATH)
print("Saved checkpoint:", CKPT_PATH)
